# Supp Table 1 — per-sample performance (SPATNIC)

**🟢 light (reads caches)** · source: `notebooks/metrics_pooled_persample_all.py`

Same script as Table 2.

## Configuration — edit the paths, then run

In [ ]:
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")
try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print("[note] scanpy/anndata not available:", e)

# ── EDIT THESE PATHS to match your environment ──
REPO_ROOT     = Path("/path/to/spatnic")          # this repository
BACKUP_ROOT   = Path("/path/to/backup")           # integrate_adata_filtered.h5ad, galaxy scores, Liver meta
DATA_ROOT     = Path("/path/to/data")             # GxD concat, lung annotated, c2l refs, spatnic_models, GxD_Xenium
BENCHMARK_DB  = Path("/path/to/benchmark_db")     # Xenium/VisiumHD/MERFISH/CosMx + adata_hvg_*
VISIUMHD_ROOT = Path("/path/to/VisiumHD")         # Visium HD ADC track
WEIGHTS_DIR   = Path.home() / ".spatnic" / "weights"

# ── Derived ──
NB     = REPO_ROOT / "notebooks"
COMP   = NB / "comparison_results"
VHD    = VISIUMHD_ROOT
MODELS = DATA_ROOT / "spatnic_models"
PAPER  = REPO_ROOT / "paper"
BASE_DIR  = VHD          # VisiumHD ADC notebook global
THRESHOLD = 0.9          # overridden to 0.5 by the Fig 5 shortcut setup cell
sys.path[:0] = [str(REPO_ROOT / "scripts"), str(NB)]
if NB.exists():
    os.chdir(NB)         # extracted cells were written for cwd = notebooks/

def _tbl(csv, n=None):
    p = Path(csv)
    if not p.exists():
        print("[missing]", p); return None
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
    display(df.head(n) if n else df); return df

def _run(script, show=None, n=None):
    import subprocess
    cmd = f"python notebooks/{script}"
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    print((r.stdout or "")[-3000:])
    if r.returncode: print("STDERR:\n", (r.stderr or "")[-2000:])
    if show: _tbl(COMP / show, n)


## Regenerate (runs the real metric program)

In [ ]:
_run("metrics_pooled_persample_all.py", show="eval_confmat/metrics_persample_meanSD.csv")

## Result (current cached values)

**Per-sample (mean ± SD)** (`metrics_persample_meanSD.csv`, 3 rows)

| config | n_samples_total | n_samples_used | AUROC | AUPRC | MCC | F1 | Sensitivity | Specificity | BalancedAccuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| primary model on primary external test ( | 7 | 7 | 0.987 ± 0.015 | 0.988 ± 0.022 | 0.817 ± 0.089 | 0.952 ± 0.047 | 0.945 ± 0.034 | 0.959 ± 0.074 | 0.952 ± 0.039 |
| liver met model on liver met test (GxD,  | 3 | 3 | 0.971 ± 0.044 | 0.974 ± 0.041 | 0.877 ± 0.171 | 0.947 ± 0.080 | 0.950 ± 0.076 | 0.927 ± 0.096 | 0.938 ± 0.086 |
| lung met model on lung met test (GxD, 9- | 1 | 1 | 0.998 (n=1) | 0.999 (n=1) | 0.930 (n=1) | 0.987 (n=1) | 0.977 (n=1) | 0.988 (n=1) | 0.982 (n=1) |